In [ ]:
!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install scipy
!pip install aif360
!pip install 'aif360[Reductions]'
!pip install 'aif360[inFairness]'

# Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import urllib.request

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

from aif360.algorithms.preprocessing import LFR
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import (
    load_preproc_data_adult,
    load_preproc_data_german,
    load_preproc_data_compas,
)

# Load datasets

In [ ]:
def download_aif360_datasets():
    # Define the directory structures and URLs
    base_dir = "/usr/local/lib/python3.13/dist-packages/aif360/data/raw"

    datasets_info = {
        "adult": [
            ("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data", "adult.data"),
            ("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test", "adult.test"),
            ("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names", "adult.names")
        ],
        "german": [
            ("https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data", "german.data"),
            ("https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.doc", "german.doc")
        ],
        "compas": [
            ("https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv", "compas-scores-two-years.csv")
        ]
    }

    for folder, files in datasets_info.items():
        target_folder = os.path.join(base_dir, folder)
        os.makedirs(target_folder, exist_ok=True)
        for url, filename in files:
            target_path = os.path.join(target_folder, filename)
            if not os.path.exists(target_path):
                print(f"Downloading {filename} to {target_folder}...")
                try:
                    urllib.request.urlretrieve(url, target_path)
                except Exception as e:
                    print(f"Failed to download {filename}: {e}")
            else:
                print(f"{filename} already exists.")

# Configure datasets and protected attributes


In [ ]:
# AIF360's helper functions define the protected group as value 1 and
# unprotected group as value 0 for these selected attributes.
DATASETS = {
    "Adult": {
        "loader": lambda: load_preproc_data_adult(protected_attributes=["sex"]),
        "protected_attribute": "sex",
        "unprivileged_groups": [{"sex": 0.0}],  # Female
        "privileged_groups": [{"sex": 1.0}],    # Male
    },
    "German": {
        "loader": lambda: load_preproc_data_german(protected_attributes=["sex"]),
        "protected_attribute": "sex",
        "unprivileged_groups": [{"sex": 0.0}],  # Female
        "privileged_groups": [{"sex": 1.0}],    # Male
    },
    "COMPAS": {
        "loader": lambda: load_preproc_data_compas(protected_attributes=["race"]),
        "protected_attribute": "race",
        "unprivileged_groups": [{"race": 0.0}],  # Not Caucasian
        "privileged_groups": [{"race": 1.0}],    # Caucasian
    },
}

# Define fairness evaluation function


In [49]:
def fairness_metrics(dataset, y_pred=None, scores=None,
                    unprivileged_groups=None, privileged_groups=None):
    """Compute DP difference, disparate impact and AIF360 consistency."""
    ds = dataset.copy(deepcopy=True)

    # If predictions are provided, replace the original labels
    # with the model's predicted labels.
    if y_pred is not None:
        y_pred = np.asarray(y_pred).astype(float).reshape(-1, 1)
        ds.labels = y_pred

        # If probability scores are not provided, use binary predictions.
        if scores is None:
            ds.scores = y_pred.copy()
        else:
            ds.scores = np.asarray(scores).astype(float).reshape(-1, 1)

    # This object compares the outcomes of the two protected groups.
    metric = BinaryLabelDatasetMetric(
        ds,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
    )

    return {
        "DP_Diff": float(metric.statistical_parity_difference()),
        "DIR": float(metric.disparate_impact()),
        "Consistency": float(np.mean(metric.consistency(n_neighbors=N_NEIGHBORS))),
    }

# Define predictive performance evaluation function


In [ ]:
def predictive_metrics(y_true, y_pred):
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Balanced_Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

# Run the experiment on one dataset


In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.30
N_NEIGHBORS = 5

In [47]:
def run_one_dataset(dataset_name, config):

    print(f"\n===== {dataset_name} =====")

    # --------------------------------------------------------
    # 1. Load dataset
    # --------------------------------------------------------

    dataset = config["loader"]()
    y = dataset.labels.ravel().astype(int)
    indices = np.arange(len(y))

    # --------------------------------------------------------
    # 2. Evaluate original fairness
    # --------------------------------------------------------

    # This serves as a reference before applying bias mitigation.
    original = fairness_metrics(
        dataset,
        unprivileged_groups=config["unprivileged_groups"],
        privileged_groups=config["privileged_groups"],
    )

    # --------------------------------------------------------
    # 3. Split and scale data
    # --------------------------------------------------------

    # Use stratified sampling to preserve the class distribution.
    train_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    train_ds = dataset.subset(train_idx)
    test_ds = dataset.subset(test_idx)

    # Extract features and labels from both subsets.
    X_train = np.asarray(train_ds.features, dtype=float)
    X_test = np.asarray(test_ds.features, dtype=float)
    y_train = train_ds.labels.ravel().astype(int)
    y_test = test_ds.labels.ravel().astype(int)

    # Scale features using training data only to prevent data leakage.
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --------------------------------------------------------
    # 4. Train baseline model
    # --------------------------------------------------------

    # Train Logistic Regression without any bias-mitigation technique.
    baseline = LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    )
    baseline.fit(X_train_scaled, y_train)

    # Generate predictions and favorable-outcome probabilities.
    baseline_pred = baseline.predict(X_test_scaled)
    baseline_score = baseline.predict_proba(X_test_scaled)[:, 1]

    # --------------------------------------------------------
    # 5. Evaluate baseline model
    # --------------------------------------------------------

    # Evaluate fairness across protected groups using test predictions.
    baseline_fairness = fairness_metrics(
        test_ds,
        y_pred=baseline_pred,
        scores=baseline_score,
        unprivileged_groups=config["unprivileged_groups"],
        privileged_groups=config["privileged_groups"],
    )

    # Evaluate predictive performance against actual test labels.
    baseline_performance = predictive_metrics(
        y_test, baseline_pred
    )

    # --------------------------------------------------------
    # 6. Apply LFR transformation
    # --------------------------------------------------------

    # Prepare AIF360 datasets with scaled features and original metadata.
    lfr_train_input = train_ds.copy(deepcopy=True)
    lfr_train_input.features = X_train_scaled

    lfr_test_input = test_ds.copy(deepcopy=True)
    lfr_test_input.features = X_test_scaled

    # Initialize LFR with reconstruction, prediction and fairness weights.
    lfr = LFR(
        unprivileged_groups=config["unprivileged_groups"],
        privileged_groups=config["privileged_groups"],
        k=5,
        Ax=0.1,
        Ay=1.0,
        Az=50.0,
        verbose=0,
        seed=RANDOM_STATE,
    )

    # Fit LFR on training data only to avoid data leakage.
    lfr.fit(lfr_train_input, maxiter=500, maxfun=500)

    # Transform training and testing data using the learned representation.
    train_transformed = lfr.transform(
        lfr_train_input, threshold=0.5
    )
    test_transformed = lfr.transform(
        lfr_test_input, threshold=0.5
    )

    # Extract transformed features and training labels.
    X_train_lfr = train_transformed.features
    X_test_lfr = test_transformed.features
    y_train_lfr = train_transformed.labels.ravel().astype(int)

    # --------------------------------------------------------
    # 7. Train and evaluate LFR model
    # --------------------------------------------------------

    # Fallback mechanism if LFR labels collapsed to a single class
    if len(np.unique(y_train_lfr)) < 2:
        print(f"[Warning] LFR labels collapsed to a single class in {dataset_name}. Using original train labels instead.")
        y_train_to_use = y_train
    else:
        y_train_to_use = y_train_lfr

    # Train the same classifier using LFR-transformed training data.
    mitigated_model = LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    )
    mitigated_model.fit(X_train_lfr, y_train_to_use)

    # Generate predictions from transformed test features.
    mitigated_pred = mitigated_model.predict(X_test_lfr)
    mitigated_score = mitigated_model.predict_proba(X_test_lfr)[:, 1]

    # Evaluate fairness using the transformed test representation.
    mitigated_fairness = fairness_metrics(
        test_transformed,
        y_pred=mitigated_pred,
        scores=mitigated_score,
        unprivileged_groups=config["unprivileged_groups"],
        privileged_groups=config["privileged_groups"],
    )

    # Compare predictions with the original ground-truth test labels.
    mitigated_performance = predictive_metrics(
        y_test, mitigated_pred
    )

    # --------------------------------------------------------
    # 8. Collect and display results
    # --------------------------------------------------------

    # Combine fairness and predictive metrics from all stages.
    rows = [
        {
            "Dataset": dataset_name,
            "Stage": "Original labels (full data)",
            **original
        },
        {
            "Dataset": dataset_name,
            "Stage": "Baseline predictions (test)",
            **baseline_fairness,
            **baseline_performance
        },
        {
            "Dataset": dataset_name,
            "Stage": "LFR predictions (test)",
            **mitigated_fairness,
            **mitigated_performance
        }
    ]

    result = pd.DataFrame(rows)
    print(result.to_string(index=False))
    return result

# Main function: Run experiments on all datasets


In [45]:
def main():
    all_results = []

    # Iterate over Adult, German Credit and COMPAS.
    for name, config in DATASETS.items():
        try:
            result = run_one_dataset(name, config)
            all_results.append(result)
        except Exception as exc:
            print(f"\nERROR while processing {name}: {exc}")
            print(
                "Check that the AIF360 raw dataset files are installed in "
                "the expected aif360/data/raw/ folders and that your AIF360 "
                "version exposes the preprocessing helper loaders."
            )
            raise

    final = pd.concat(all_results, ignore_index=True)
    final.to_csv("lfr_fairness_results.csv", index=False)
    print("\nSaved: lfr_fairness_results.csv")

# Execute the experiment


In [50]:
if __name__ == "__main__":
    try:
        download_aif360_datasets()
    except Exception as e:
        print(f"Error setting up datasets: {e}")

    # Run the main experiment
    main()

adult.data already exists.
adult.test already exists.
adult.names already exists.
german.data already exists.
german.doc already exists.
compas-scores-two-years.csv already exists.

===== Adult =====
Dataset                       Stage   DP_Diff      DIR  Consistency  Accuracy       F1  Balanced_Accuracy
  Adult Original labels (full data) -0.194516 0.359655     0.719319       NaN      NaN                NaN
  Adult Baseline predictions (test) -0.207636 0.000000     0.999918  0.803180 0.479422           0.657722
  Adult      LFR predictions (test)  0.031090 1.181937     0.999864  0.646011 0.158091           0.472206

===== German =====
[Warning] LFR labels collapsed to a single class in German. Using original train labels instead.
Dataset                       Stage   DP_Diff      DIR  Consistency  Accuracy       F1  Balanced_Accuracy
 German Original labels (full data) -0.074801 0.896567       0.6614       NaN      NaN                NaN
 German Baseline predictions (test) -0.187671 0